# Rossmann Store Sales Project

## Phase 5: Feature Engineering & Modeling Build

### Import Essential Libraries

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import seaborn as sns

# Set display options to see all columns clearly
pd.set_option('display.max_columns', None)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


### Load Cleaned Data

In [10]:
# Load the cleaned dataset from the data folder
train_filtered = pd.read_csv('../data/processed/train_cleaned.csv', low_memory=False)

### Quick Verification Check

In [11]:
# Check the first few rows and data types
display(train_filtered.head())

print(train_filtered.info())

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,Month,Year
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,0.0,0.0,NaN,7,2015
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct",7,2015
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct",7,2015
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,0.0,0.0,NaN,7,2015
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,0.0,0.0,NaN,7,2015


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 844338 entries, 0 to 844337
Data columns (total 20 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Store                      844338 non-null  int64  
 1   DayOfWeek                  844338 non-null  int64  
 2   Date                       844338 non-null  object 
 3   Sales                      844338 non-null  int64  
 4   Customers                  844338 non-null  int64  
 5   Open                       844338 non-null  int64  
 6   Promo                      844338 non-null  int64  
 7   StateHoliday               844338 non-null  object 
 8   SchoolHoliday              844338 non-null  int64  
 9   StoreType                  844338 non-null  object 
 10  Assortment                 844338 non-null  object 
 11  CompetitionDistance        844338 non-null  float64
 12  CompetitionOpenSinceMonth  844338 non-null  float64
 13  CompetitionOpenSinceYear   84

### Drop the Date column

In [13]:
# Drop the raw Date column (numerical components already extracted), skipping safely to prevent errors on notebook re-runs
train_filtered = train_filtered.drop(columns=['Date'], errors='ignore')
print("Date column dropped successfully.")

Date column dropped successfully.


### Check Remaining Categorical Columns and Identify Which Need Encoding

In [14]:
# Identify categorical columns
categorical_cols = train_filtered.select_dtypes(include=['object', 'category']).columns
print("Categorical columns:", list(categorical_cols))

# Check unique values for each true categorical column
for col in categorical_cols:
    print(f"Column '{col}' unique values:", train_filtered[col].unique())

Categorical columns: ['StateHoliday', 'StoreType', 'Assortment', 'PromoInterval']
Column 'StateHoliday' unique values: ['0' 'a' 'b' 'c']
Column 'StoreType' unique values: ['c' 'a' 'd' 'b']
Column 'Assortment' unique values: ['a' 'c' 'b']
Column 'PromoInterval' unique values: [nan 'Jan,Apr,Jul,Oct' 'Feb,May,Aug,Nov' 'Mar,Jun,Sept,Dec']


### Encode Categorical Variables

#### Choosing an Encoding Strategy: Explicit Mapping vs. `.cat.codes`

* **Explicit Mapping (e.g., `StateHoliday`):** Use when specific numbers carry strict meaning (e.g., `0` = no holiday) or when handling mixed data types. It acts as a permanent rulebook that prevents category values from shifting or mismatching between your train and test sets.
* **`.cat.codes` (e.g., `StoreType`, `Assortment`):** Use for clean, uniform text labels where exact numeric values don't matter. It is a quick, convenient way to convert categories into integers for tree-based models.

In [15]:
# 1. Map StateHoliday explicitly to preserve consistent meaning and handle mixed types
state_holiday_mapping = {'0': 0, 0: 0, 'a': 1, 'b': 2, 'c': 3}
train_filtered['StateHoliday'] = train_filtered['StateHoliday'].map(state_holiday_mapping)

# 2. Convert StoreType and Assortment to category codes efficiently
train_filtered['StoreType'] = train_filtered['StoreType'].astype('category').cat.codes
train_filtered['Assortment'] = train_filtered['Assortment'].astype('category').cat.codes

print("Categorical encoding for StateHoliday, StoreType, and Assortment completed successfully!")

Categorical encoding for StateHoliday, StoreType, and Assortment completed successfully!


 i need to check nan value in PromoInterval